# 04 - AD over a TPSA map

TPSA differentiates with respect to phase-space coordinates. Enzyme can additionally differentiate a selected map coefficient with respect to a lattice parameter.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(joinpath(EXAMPLES_DIR, "environments", "tpsa_ad"))
Pkg.instantiate()
using TrackPad, PolySeries, Enzyme
include(joinpath(EXAMPLES_DIR, "common.jl")); using .TrackPadExamples

variables = polyseries_variables(Float64; order=2)  # descriptor setup stays outside AD
base_ring, beam = madx_fodo()

sf_index = findfirst(e -> e.name == :sf, base_ring.elements)
sf_template = base_ring[sf_index]
left_drift = base_ring[sf_index - 1]
right_drift = base_ring[sf_index + 1]
left_L, sf_L = left_drift.L, sf_template.L
sf_steps, right_L = sf_template.num_int_steps, right_drift.L

In [ ]:
function geometric_term(k2::Float64, left_L::Float64, sf_L::Float64,
                        sf_steps::Int, right_L::Float64, variables, beam)
    section = Lattice(AbstractElement[
        Drift(left_L),
        Sextupole(sf_L, k2; num_int_steps=sf_steps),
        Drift(right_L),
    ])
    map = linepass(section, variables, beam)
    return element(map[2], [2, 0, 0, 0, 0, 0])
end

function scalar_derivative(raw)
    value = raw isa Tuple ? raw[1] : raw
    while !(value isa Number)
        value = value isa NamedTuple ? first(values(value)) : value[1]
    end
    return value
end

k2 = sf_template.k2
mode = Enzyme.set_runtime_activity(Enzyme.Forward)
raw = Enzyme.autodiff(
    mode, Enzyme.Const(geometric_term), Enzyme.Duplicated,
    Enzyme.Duplicated(k2, 1.0),
    Enzyme.Const(left_L), Enzyme.Const(sf_L), Enzyme.Const(sf_steps),
    Enzyme.Const(right_L), Enzyme.Const(variables), Enzyme.Const(beam),
)
derivative = scalar_derivative(raw)
value = geometric_term(k2, left_L, sf_L, sf_steps, right_L, variables, beam)
(; value, derivative)

In [ ]:
h = 1e-5
finite_difference = (
    geometric_term(k2 + h, left_L, sf_L, sf_steps, right_L, variables, beam) -
    geometric_term(k2 - h, left_L, sf_L, sf_steps, right_L, variables, beam)
) / (2h)
error = derivative - finite_difference
relative_error = abs(error) / max(abs(derivative), abs(finite_difference), eps())
(; derivative, finite_difference,
   agreement=isapprox(derivative, finite_difference; rtol=1e-8, atol=1e-10),
   abs_error=abs(error), relative_error)

Call `polyseries_variables` (or `set_descriptor!`) outside the differentiated function. Select only the coefficient needed by the design objective rather than differentiating the complete coefficient storage. Forward mode is intentional: reverse mode through a complete TPSA lattice map currently produces an impractically large LLVM compile. Expect roughly a minute for the first forward-mode compilation; subsequent calls in the same Julia session reuse it.